# Gustafson's Law & Weak Scaling — Complete Practice Notebook
**Student:** Yerassyl Orazgali | **ID:** 230103272

Open this notebook in Google Colab and choose **Runtime → Run all**. It creates a repository-style folder with theory reports, calculations, benchmark code/data, analysis, plots, and README. Measurements are generated on the active Colab runtime, not fabricated.

In [ ]:
from pathlib import Path
import os, sys, time, math, statistics, platform, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STUDENT_NAME = "Yerassyl Orazgali"
STUDENT_ID = "230103272"
ROOT = Path("/content/gustafson-law-practice")
for d in ["theory", "calculations", "benchmarks/weak_scale_bench", "analysis", "plots"]:
    (ROOT/d).mkdir(parents=True, exist_ok=True)

def write(folder, filename, content):
    (ROOT/folder/filename).write_text(content.strip()+"\\n", encoding="utf-8")

def gustafson(P, alpha): return P - alpha*(P-1)
def efficiency(P, alpha): return gustafson(P, alpha)/P
def amdahl(P, s): return 1/(s+(1-s)/P)

print("Ready:", STUDENT_NAME, STUDENT_ID, "| CPUs:", os.cpu_count())

## Part I — Theoretical foundations (Tasks 01–05)

In [ ]:
theory = {
"task01_scaling_models.md": r"""
# Task 01 — Strong Scaling vs. Weak Scaling
**Student:** Yerassyl Orazgali (230103272)

**Amdahl (1967):** Amdahl assumes a fixed total problem size. If `s` is the serial fraction and `1-s` is parallel, then
\[
S_A(P)=\frac{1}{s+(1-s)/P}.
\]
As \(P\to\infty\), the total workload and problem dimension remain fixed and \(S_A\to1/s\). This is fixed-size or strong scaling.

**Gustafson (1988):** Gustafson assumes approximately fixed parallel execution time while the workload grows with processor count:
\[
S_G(P)=\alpha+(1-\alpha)P=P-\alpha(P-1).
\]
As \(P\to\infty\), the problem dimension/workload is allowed to increase with aggregate processing capacity. This is fixed-time or weak scaling.

Amdahl asks how fast the same job can finish; Gustafson asks how much larger a job can finish in about the same time. Their assumptions differ, so their conclusions are not contradictory.
""",
"task02_derivation.md": r"""
# Task 02 — Derivation of Gustafson's Scaled Speedup
**Student:** Yerassyl Orazgali (230103272)

Normalize parallel runtime to \(T_P=s+p=1\), where \(s\) is serial time and \(p=1-s\) is parallel time. In the same elapsed time, the equivalent serial machine performs the serial work \(s\) plus the parallel work on all \(P\) processors, \(Pp\). Thus
\[
T_1=s+Pp,\qquad S_G(P)=T_1/T_P=s+Pp.
\]
Substitute \(p=1-s\):
\[
S_G=s+P(1-s)=P-s(P-1).
\]
Writing the observed serial fraction as \(\alpha=s\) gives
\[
\boxed{S_G(P)=P-\alpha(P-1)=\alpha+(1-\alpha)P.}
\]
The derivation assumes the parallel elapsed time stays normalized and the workload expands with \(P\).
""",
"task03_serial_invariance.md": r"""
# Task 03 — Serial Fraction Measurement Frame
**Student:** Yerassyl Orazgali (230103272)

Amdahl's serial fraction is usually defined relative to a fixed workload's uniprocessor execution. Gustafson's \(\alpha\) is the serial share observed on the parallel machine while executing a scaled workload. The fractions therefore use different frames of reference and need not be equal.

With parallel runtime normalized to \(s+p=1\), equivalent serial work is \(s+Pp\), yielding \(S_G=s+Pp\). The apparent paradox disappears: Amdahl holds total work constant, while Gustafson holds parallel elapsed time approximately constant and increases useful work. Serial overhead is not free; communication and synchronization can make the observed \(\alpha\) increase with \(P\).
""",
"task04_asymptotic_limits.md": r"""
# Task 04 — Asymptotic Parallel Efficiency
**Student:** Yerassyl Orazgali (230103272)

For Gustafson:
\[
E_G(P)=S_G(P)/P=1-\alpha+\alpha/P,
\quad \lim_{P\to\infty}E_G(P)=1-\alpha.
\]
For Amdahl:
\[
E_A(P)=S_A(P)/P=\frac{1}{sP+(1-s)}.
\]
For any fixed \(s>0\), \(\lim_{P\to\infty}E_A(P)=0\). If \(s=0\), Amdahl efficiency remains 1. Gustafson approaches a nonzero limit for \(\alpha<1\) because the workload grows with processor count.
""",
"task05_historical_ncube.md": r"""
# Task 05 — Historical nCUBE Example
**Student:** Yerassyl Orazgali (230103272)

The practice sheet states that Gustafson, Montry, and Benner reported speedups greater than 1,000× on a 1,024-processor nCUBE 10, and gives a 0.4% serial fraction. Under Amdahl's fixed-size model, \(s=0.004\) gives an asymptotic ceiling \(1/s=250\).

Under Gustafson's scaled-workload model:
\[
S_G=1024-0.004(1023)=1019.908.
\]
This exceeds 1,000 and matches the scale of the result described in the task sheet. Problem expansion explains the difference: more processors execute a larger useful workload in roughly the same time, rather than merely dividing an unchanged job.

**Scope:** The supplied sheet does not include the original workload name, raw measurements, or detailed nCUBE hardware configuration. Those details are not invented in this report.
"""
}
for f, t in theory.items(): write("theory", f, t)
print("Created Tasks 01–05.")

## Part II — Numerical calculations (Tasks 06–10)

In [ ]:
# Task 06
alpha = 0.04
ps = [16,64,256,1024,4096,16384]
t6 = pd.DataFrame({"P":ps,
 "Gustafson_speedup":[gustafson(p,alpha) for p in ps],
 "Gustafson_efficiency":[efficiency(p,alpha) for p in ps]})
write("calculations","task06_parametric_table.md",
      "# Task 06 — Parametric Speedup\\n\\nα = 0.04\\n\\n"+t6.to_markdown(index=False))
display(t6)

# Task 07
P, target = 8192, 7500
amax = (P-target)/(P-1)
write("calculations","task07_tolerable_serial.md", f"""
# Task 07 — Maximum Tolerable Serial Fraction
Given S(P)=P−α(P−1) ≥ {target} and P={P}:
α ≤ (P−{target})/(P−1) = ({P}-{target})/({P}-1).
Therefore α_max = {amax:.10f} = {amax*100:.6f}%.
""")
print("Task 07 alpha_max =", amax)

# Task 08
P,s=128,0.08
sa,sg=amdahl(P,s),gustafson(P,s)
disc=(sg-sa)/sa*100
write("calculations","task08_cross_divergence.md", f"""
# Task 08 — Amdahl vs. Gustafson
Amdahl: 1/(s+(1−s)/P) = {sa:.8f}.
Gustafson: P−α(P−1) = {sg:.8f}.
Gustafson is {disc:.4f}% higher, using Amdahl as the denominator.
They diverge because Amdahl fixes total work while Gustafson expands it.
""")
print("Task 08:", sa, sg, disc, "%")

# Task 09
a9=0.02+0.06
t9=pd.DataFrame({"P":[512,2048],"aggregate_alpha":[a9],
                 "Gustafson_speedup":[gustafson(512,a9),gustafson(2048,a9)]})
write("calculations","task09_multicomponent.md",
      "# Task 09 — Multi-Component Workload\\nI/O/broadcast 2% + synchronization 6% = α=8%; compute=92%.\\n\\n"+t9.to_markdown(index=False))
display(t9)

# Task 10: explicit assumption, one 8-byte value per grid cell
rows=[]
for nodes in [64,512]:
    total_bytes=nodes*32*(1024**3)
    cells=total_bytes/8
    rows.append({"nodes":nodes,"total_memory_GB":nodes*32,
                 "ideal_cells":int(cells),"cubic_dimension_N":cells**(1/3)})
t10=pd.DataFrame(rows)
write("calculations","task10_memory_growth.md",
      "# Task 10 — Memory Growth\\nAssumption: one 8-byte value per cell; all memory is available to this array. No halo, temporary arrays, metadata, or OS memory included, so values are ideal upper bounds.\\n\\n"+t10.to_markdown(index=False))
display(t10)

## Part III — Comparative scaling (Tasks 11–15)

In [ ]:
more_theory = {
"task11_sun_ni_comparison.md": r"""
# Task 11 — Gustafson vs. Sun–Ni Memory-Bounded Speedup
**Student:** Yerassyl Orazgali (230103272)

Gustafson scales workload to keep parallel execution time approximately fixed:
\[
S_G(P)=\alpha+(1-\alpha)P.
\]
Sun–Ni generalizes scaling by considering growth in available memory capacity. Achievable problem size depends on memory growth and algorithm memory requirements, not only processor count. Gustafson can overestimate throughput when memory capacity/bandwidth fails to grow, or communication and synchronization rise. It can underestimate throughput when memory capacity and data locality permit efficient growth beyond a simple processor-only model. The practice sheet does not specify a particular Sun–Ni parameterization, so a universal closed-form equation cannot be inferred from it.
""",
"task12_communication_penalty.md": r"""
# Task 12 — Communication Penalty Extension
**Student:** Yerassyl Orazgali (230103272)

Let \(\alpha_0\) be non-communication serial work and \(C(P)\ge0\) be normalized communication overhead. One illustrative extension is
\[
S_{G,C}(P)=\frac{\alpha_0+(1-\alpha_0)P}{1+C(P)}.
\]
Examples: \(C(P)=\beta\log_2P\) for logarithmic overhead, or \(C(P)=\gamma\sqrt P\) for a surface/mesh-like penalty, with measured constants \(\beta,\gamma\ge0\). If \(C(P)=0\), ordinary Gustafson speedup is recovered. Growing communication overhead makes speedup sublinear. Avoid double-counting communication if it is already included in \(\alpha_0\).
""",
"task13_failure_cases.md": r"""
# Task 13 — When Weak Scaling Fails
**Student:** Yerassyl Orazgali (230103272)

1. **Direct all-pairs N-body:** Each iteration has \(O(N^2)\) interactions. Increasing particle count can raise work and remote particle-data exchange; communication and synchronization can grow, so simply adding processors does not guarantee constant runtime.
2. **BFS on power-law graphs:** Frontier sizes are data-dependent. High-degree vertices create contention, irregular memory access, load imbalance, and communication hot spots. Equal vertex counts do not imply equal work.

Both cases violate the ideal premise when overhead or imbalance grows with system size.
""",
"task14_isoefficiency.md": r"""
# Task 14 — Isoefficiency under Gustafson Scaling
**Student:** Yerassyl Orazgali (230103272)

Isoefficiency describes how total work \(W(P)\) must grow to maintain a target efficiency. In ideal weak scaling, work per processor is constant:
\[
W(P)=P\,W(1)=\Theta(P).
\]
If the parallel runtime and overhead fraction remain bounded, then
\[
E_G(P)=1-\alpha+\alpha/P\to1-\alpha.
\]
Thus a Gustafson-compliant ideal workload has linear isoefficiency \(W(P)=\Theta(P)\) under these assumptions. Communication or synchronization can change this relation or prevent constant efficiency.
""",
"task15_energy_scaling.md": r"""
# Task 15 — Energy and Power under Weak Scaling
**Student:** Yerassyl Orazgali (230103272)

If each of \(P\) nodes draws active power \(P_{active}\) for runtime \(t\), then
\[
E_P=P P_{active}t,\qquad E_1=P_{active}t,\qquad E_P/E_1=P.
\]
If idle and dynamic components are separated, per-node power is \(P_{idle}+(P_{active}-P_{idle})=P_{active}\) while active. Thus wall-clock time can remain approximately fixed while total energy grows roughly linearly with node count. This simplified model excludes cooling, network/storage energy, sleep states, and variable utilization.
"""
}
for f,t in more_theory.items(): write("theory",f,t)
print("Created Tasks 11–15.")

## Part IV — Domain-specific workloads (Tasks 16–20)

In [ ]:
domain = {
"task16_gemm_scaling.md": r"""
# Task 16 — Dense Matrix Multiplication (GEMM)
**Student:** Yerassyl Orazgali (230103272)

Dense multiplication requires \(\Theta(N^3)\) operations and \(\Theta(N^2)\) storage. For total useful work proportional to processor count:
\[
N^3\propto P\Rightarrow N\propto P^{1/3}.
\]
Memory then grows as \(N^2\propto P^{2/3}\). Communication, data movement, cache capacity, and the chosen parallel algorithm can still prevent constant runtime.
""",
"task17_stencil_scaling.md": r"""
# Task 17 — 3D 7-Point Stencil
**Student:** Yerassyl Orazgali (230103272)

For cubic side length \(N\), interior computation is \(\Theta(N^3)\), while boundary exchange is \(\Theta(N^2)\). Therefore communication/computation is \(\Theta(1/N)\). If global volume grows proportionally to \(P\), global dimension grows as \(P^{1/3}\) and global surface/volume scales as \(P^{-1/3}\). Under strict weak decomposition with fixed local subdomain size per node, each node's local surface/volume ratio remains approximately constant. Global and per-node ratios must not be confused.
""",
"task18_nbody_scaling.md": r"""
# Task 18 — Direct N-Body vs. Barnes–Hut
**Student:** Yerassyl Orazgali (230103272)

Direct all-pairs N-body costs \(\Theta(N^2)\). Under ideal aggregate compute growth proportional to \(P\), fixed iteration time suggests \(N^2\propto P\), or \(N\propto P^{1/2}\). Barnes–Hut costs approximately \(\Theta(N\log N)\); constant time suggests \(N\log N\propto P\), implicitly giving growth roughly \(P/\log P\). These compute-only estimates omit communication, tree construction, traversal, and load imbalance, so actual serial/parallel ratios are implementation-dependent.
""",
"task19_monte_carlo.md": r"""
# Task 19 — Monte Carlo Weak Scaling
**Student:** Yerassyl Orazgali (230103272)

Monte Carlo paths are mostly independent. Nonzero serial/coordination work includes PRNG stream setup, initialization, reductions, and output serialization. A balanced binary reduction over \(P\) partial results has \(\lceil\log_2P\rceil\) communication stages and \(P-1\) combine operations. If each stage costs \(\tau\), critical-path latency is approximately \(\tau\lceil\log_2P\rceil\), plus data transfer. This overhead can raise observed \(\alpha\) or be modeled separately as \(C(P)\).
""",
"task20_graph_processing.md": r"""
# Task 20 — BFS/PageRank and Irregular Graphs
**Student:** Yerassyl Orazgali (230103272)

BFS and PageRank on power-law graphs have irregular memory access, data-dependent frontiers, skewed vertex degrees, load imbalance, and synchronization/reduction costs. Partition boundaries create network traffic and high-degree vertices can become hot spots. Dense linear algebra is more regular, with predictable blocks and mature communication-avoiding algorithms. Adding processors to graph analytics may therefore add overhead without proportionally increasing useful throughput.
"""
}
for f,t in domain.items(): write("analysis",f,t)
print("Created Tasks 16–20.")

## Part V — Simulator and empirical benchmark (Tasks 21–24)

In [ ]:
# Task 21: simulator CSV and comparison plot
Pvals=np.unique(np.logspace(0,np.log10(65536),100).astype(int))
alphas=[0.001,0.01,0.04,0.10,0.20]
sim=[]
for a in alphas:
    for p in Pvals:
        sim.append({"P":int(p),"alpha":a,"Gustafson":gustafson(int(p),a),
                    "Amdahl":amdahl(int(p),a),"Gustafson_efficiency":efficiency(int(p),a)})
simdf=pd.DataFrame(sim)
simdf.to_csv(ROOT/"benchmarks/simulator_output.csv",index=False)
fig,ax=plt.subplots(figsize=(10,6))
for a in alphas:
    q=simdf[simdf.alpha==a]
    ax.plot(q.P,q.Gustafson,label=f"Gustafson α={a}")
    ax.plot(q.P,q.Amdahl,"--",label=f"Amdahl s={a}")
ax.set_xscale("log",base=2); ax.set_yscale("log")
ax.set(xlabel="Processor count P",ylabel="Speedup",title="Amdahl vs. Gustafson")
ax.grid(True,which="both",alpha=.3); ax.legend(ncol=2,fontsize=8)
fig.tight_layout(); fig.savefig(ROOT/"plots/task21_scaling_simulator.png",dpi=200); plt.show()
print("Task 21 CSV and plot saved.")

In [ ]:
# Tasks 22–23: multiprocessing weak-scaling benchmark, 5 runs per worker count
import multiprocessing as mp

def worker_job(args):
    n, seed = args
    rng=np.random.default_rng(seed)
    x=rng.random(n)
    return float(np.sum(np.sin(x)*np.cos(x)))

def benchmark_case(n_per_worker, workers, repeats=5):
    times=[]; checksum=0.0
    for r in range(repeats):
        jobs=[(n_per_worker,10000+r*workers+i) for i in range(workers)]
        start=time.perf_counter()
        with mp.Pool(workers) as pool:
            values=pool.map(worker_job,jobs)
        times.append(time.perf_counter()-start)
        checksum=sum(values)
    return times,checksum

cpu_count=os.cpu_count() or 2
worker_counts=sorted(set([1,2,4,min(8,cpu_count),min(16,cpu_count)]))
worker_counts=[w for w in worker_counts if 1<=w<=cpu_count]
WORK_PER_WORKER=250_000
REPEATS=5
print("Platform:",platform.platform())
print("Processor:",platform.processor() or "Not reported")
print("Visible CPUs:",cpu_count,"Workers:",worker_counts)
records=[]
for w in worker_counts:
    ts,check=benchmark_case(WORK_PER_WORKER,w,REPEATS)
    records.append({"workers":w,"work_per_worker":WORK_PER_WORKER,
                    "total_items":w*WORK_PER_WORKER,
                    "runs_seconds":";".join(f"{v:.8f}" for v in ts),
                    "mean_seconds":statistics.mean(ts),
                    "stdev_seconds":statistics.stdev(ts) if len(ts)>1 else 0,
                    "checksum":check})
results=pd.DataFrame(records)
results.to_csv(ROOT/"benchmarks/results.csv",index=False)
display(results)

In [ ]:
# Task 23: least-squares estimate of observed alpha
t1=float(results.loc[results.workers==results.workers.min(),"mean_seconds"].iloc[0])
results["empirical_speedup"]=t1/results.mean_seconds
x=results.workers.to_numpy(float)-1
y=results.workers.to_numpy(float)-results.empirical_speedup.to_numpy(float)
alpha_obs=float(np.dot(x,y)/np.dot(x,x)) if np.dot(x,x)>0 else 0.0
alpha_obs=min(1.0,max(0.0,alpha_obs))
results.to_csv(ROOT/"benchmarks/empirical_speedup.csv",index=False)
fit_code = """import pandas as pd, numpy as np
df=pd.read_csv('results.csv')
t1=float(df.loc[df.workers==df.workers.min(),'mean_seconds'].iloc[0])
df['empirical_speedup']=t1/df['mean_seconds']
x=df.workers.to_numpy(float)-1
y=df.workers.to_numpy(float)-df.empirical_speedup.to_numpy(float)
alpha=float(np.dot(x,y)/np.dot(x,x)) if np.dot(x,x)>0 else 0.0
print('Least-squares alpha:', min(1.0,max(0.0,alpha)))
"""
write("benchmarks","fit_gustafson.py",fit_code)
display(results[["workers","mean_seconds","stdev_seconds","empirical_speedup"]])
print("Observed alpha (least squares):",alpha_obs)

In [ ]:
# Task 24: measured vs theoretical plot and hardware audit
pmax=max(worker_counts)
px=np.linspace(1,pmax,200)
fig,ax=plt.subplots(figsize=(9,6))
ax.plot(px,px,label="Ideal linear")
ax.plot(px,gustafson(px,alpha_obs),label=f"Gustafson fit α={alpha_obs:.4f}")
ax.plot(px,amdahl(px,alpha_obs),"--",label=f"Amdahl s={alpha_obs:.4f}")
err=results.stdev_seconds/results.mean_seconds*results.empirical_speedup
ax.errorbar(results.workers,results.empirical_speedup,yerr=err,fmt="o",capsize=4,label="Measured")
ax.set(xlabel="Workers",ylabel="Speedup vs. smallest worker count",
       title="Empirical vs. theoretical scaling")
ax.grid(True,alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(ROOT/"plots/task24_empirical_comparison.png",dpi=220); plt.show()

audit=f"""
# Task 24 — Hardware and Runtime Audit
**Student:** {STUDENT_NAME} ({STUDENT_ID})
- Platform: `{platform.platform()}`
- Processor string: `{platform.processor() or "Not reported"}`
- Logical CPUs visible: `{cpu_count}`
- Python: `{sys.version.split()[0]}`
- NumPy: `{np.__version__}`

Colab may use a virtualized/shared host. Exact CPU model, cache hierarchy, physical memory channels, and NUMA topology are not reliably exposed here and are not guessed. Deviations may result from process startup, IPC, CPU quotas/shared-host contention, memory bandwidth, scheduling noise, and worker-to-core placement.

Error bars show runtime standard deviation propagated approximately to speedup; they are not formal confidence intervals.
"""
write("analysis","task24_hardware_audit.md",audit)
print("Saved Task 24 plot and audit.")

## Task 25 — README, artifact index, and ZIP download

In [ ]:
readme=f"""
# Gustafson's Law & Weak Scaling — Practice Repository
**Student:** {STUDENT_NAME}  
**ID:** {STUDENT_ID}

## Contents
- `theory/`: Tasks 01–05 and 11–15.
- `calculations/`: Tasks 06–10.
- `analysis/`: Tasks 16–20 and Task 24 audit.
- `benchmarks/`: Task 21 simulator, Task 22 benchmark, Task 23 results/fitting.
- `plots/`: simulator and empirical plots.

## Core formulas
Gustafson: \(S_G(P)=P-\\alpha(P-1)\), \(E_G(P)=1-\\alpha+\\alpha/P\).  
Amdahl: \(S_A(P)=1/(s+(1-s)/P)\).

## Google Colab
Open the notebook and select Runtime → Run all. It creates `/content/gustafson-law-practice/`.

## Assumptions
Task 10 assumes one 8-byte value per grid cell and ignores halo/temporary storage. Benchmark results are measured on the current Colab runtime. Task 05 uses only historical numerical details provided in the supplied practice sheet; missing experimental details are not fabricated.

## Git
```bash
git add gustafson-law-practice/
git commit -m "feat: complete Gustafson law practice tasks"
```
"""
write("","README.md",readme)
items=[]
for folder in ["theory","calculations","benchmarks","analysis","plots"]:
    items.append("## "+folder+"/")
    for f in sorted((ROOT/folder).rglob("*")):
        if f.is_file(): items.append("- `"+str(f.relative_to(ROOT))+"`")
    items.append("")
write("","ARTIFACT_INDEX.md","# Artifact Index\\n\\n"+STUDENT_NAME+" ("+STUDENT_ID+")\\n\\n"+"\\n".join(items))
zip_path=Path("/content/gustafson-law-practice.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in ROOT.rglob("*"):
        if f.is_file(): z.write(f,f.relative_to(ROOT.parent))
print("Generated files:",sum(1 for f in ROOT.rglob("*") if f.is_file()))
print("ZIP ready:",zip_path)